In [ ]:
import time
import requests

# Get this by creating a new application in the Microsoft Entra admin center
CLIENT_ID = "c36a9fb6-4f2a-41ff-90bd-ae7cc92031eb" # Prism Launcher's OAuth client ID for this text

# Also you must request access to the api from
# https://help.minecraft.net/hc/en-us/articles/16254801392141
# for your application to be able to authenticate with Minecraft services

In [ ]:
# STEP 1: For a new account, get Refresh Token, via Device Code Flow
device_code_url = "https://login.microsoftonline.com/consumers/oauth2/v2.0/devicecode"

payload = {
    "client_id": CLIENT_ID,
    "scope": "XboxLive.signin offline_access"
}

headers = {
    "Content-Type": "application/x-www-form-urlencoded"
}

response = requests.post(device_code_url, data=payload, headers=headers)

if response.status_code != 200:
    print(f"Failed to get device code: {response.text}")
    exit(1)

data = response.json()
device_code = data["device_code"]
user_code = data["user_code"]
verification_uri = data["verification_uri"]
interval = data.get("interval", 5)

# STEP 2: Show instructions to the user
print(f"1. Open your web browser and go to: {verification_uri}")
print(f"2. Enter this code: {user_code}")


In [ ]:
# STEP 2: Wait for the user to authenticate and poll for the refresh token
token_url = "https://login.microsoftonline.com/consumers/oauth2/v2.0/token"

token_payload = {
    "grant_type": "urn:ietf:params:oauth:grant-type:device_code",
    "client_id": CLIENT_ID,
    "device_code": device_code
}

print("Waiting for you to log in in your browser (polling)...")

while True:
    token_response = requests.post(
        token_url,
        data=token_payload,
        headers=headers
    )

    token_data = token_response.json()

    if token_response.status_code == 200:
        refresh_token = token_data["refresh_token"]

        print("\n[🎉] Success! Logged in successfully.")
        print("Refresh Token:", refresh_token)

        break

    error = token_data.get("error")

    if error == "authorization_pending":
        time.sleep(interval)

    elif error == "slow_down":
        interval += 5
        time.sleep(interval)

    elif error == "expired_token":
        print("\n[❌] Error: The device code expired.")
        refresh_token = None
        break

    elif error == "access_denied":
        print("\n[❌] Error: You declined the login request.")
        refresh_token = None
        break

    else:
        print(f"\n[❌] Unexpected error: {token_data}")
        refresh_token = None
        break

In [ ]:
# STEP 3: Refresh the Microsoft access token using the stored refresh token.
# run this on launcher startup so the user does not need to log in again.
# always save the new refresh token returned by Microsoft, as refresh tokens usually gets rotated.

token_url = "https://login.microsoftonline.com/consumers/oauth2/v2.0/token"

response = requests.post(
    token_url,
    data={
        "client_id": CLIENT_ID,
        "grant_type": "refresh_token",
        "refresh_token": refresh_token,
        "scope": "XboxLive.signin offline_access"
    },
    headers={
        "Content-Type": "application/x-www-form-urlencoded"
    }
)

token_data = response.json()

if response.status_code == 200:
    ms_access_token = token_data["access_token"]
    refresh_token = token_data["refresh_token"]

    # Microsoft may rotate refresh tokens
    if "refresh_token" in token_data:
        refresh_token = token_data["refresh_token"]

    print("[🎉] Microsoft access token obtained!")
    print("Access Token:", ms_access_token)
    print("Refresh Token:", refresh_token) # Update the stored refresh token with the new one

else:
    print("[❌] Failed to refresh token")
    print(token_data)

    ms_access_token = None

In [ ]:
# STEP 4: Authenticate with Xbox Live
xbl_url = "https://user.auth.xboxlive.com/user/authenticate"

xbl_payload = {
    "Properties": {
        "AuthMethod": "RPS",
        "SiteName": "user.auth.xboxlive.com",
        "RpsTicket": f"d={ms_access_token}"
    },
    "RelyingParty": "http://auth.xboxlive.com",
    "TokenType": "JWT"
}

xbl_headers = {
    "Content-Type": "application/json",
    "Accept": "application/json"
}

xbl_response = requests.post(
    xbl_url,
    json=xbl_payload,
    headers=xbl_headers
)

xbl_data = xbl_response.json()

if xbl_response.status_code == 200:
    xbl_token = xbl_data["Token"]
    user_hash = xbl_data["DisplayClaims"]["xui"][0]["uhs"]

    print("[🎉] Xbox Live authentication successful!")
    print("User Hash:", user_hash)
    print("XBL Token:", xbl_token)

else:
    print("[❌] Xbox Live authentication failed")
    print(xbl_response.status_code)
    print(xbl_data)

In [ ]:
# STEP 5: Update XSTS Token from Minecraft Services relying party
xsts_url = "https://xsts.auth.xboxlive.com/xsts/authorize"

xsts_payload = {
    "Properties": {
        "SandboxId": "RETAIL",
        "UserTokens": [
            xbl_token
        ]
    },
    "RelyingParty": "rp://api.minecraftservices.com/",
    "TokenType": "JWT"
}

xsts_headers = {
    "Content-Type": "application/json",
    "Accept": "application/json"
}

xsts_response = requests.post(
    xsts_url,
    json=xsts_payload,
    headers=xsts_headers
)

xsts_data = xsts_response.json()

if xsts_response.status_code == 200:
    xsts_token = xsts_data["Token"]

    # Update xui from the XSTS response
    xui = xsts_data["DisplayClaims"]["xui"][0]
    user_hash = xui["uhs"]

    print("[🎉] XSTS authorization successful!")
    print("User Hash:", user_hash)
    print("XSTS Token:", xsts_token)

else:
    print("[❌] XSTS authorization failed")

    # common XSTS errors related to Xbox accounts, with their corresponding error codes and messages
    if "XErr" in xsts_data:
        xerr = xsts_data["XErr"]

        if xerr == 2148916233:
            print("No Xbox profile found. Sign in to xbox.com and complete Xbox account setup.")
        elif xerr == 2148916235:
            print("Xbox Live is unavailable for this account or region.")
        elif xerr == 2148916236:
            print("Adult verification is required for this Microsoft account.")
        elif xerr == 2148916237:
            print("Age verification is required for this Microsoft account.")
        elif xerr == 2148916238:
            print("This child account must be added to a Microsoft Family by an adult.")
        else:
            print(f"Unknown XSTS error: {xerr}")

    print(xsts_response.status_code)
    print(xsts_data)

    # Common Xbox account errors
    if "XErr" in xsts_data:
        print("XErr:", xsts_data["XErr"])


In [ ]:
# STEP 6: Authenticate with Minecraft
mc_auth_url = "https://api.minecraftservices.com/authentication/login_with_xbox"
identity_token = f"XBL3.0 x={user_hash};{xsts_token}"

mc_auth_payload = {
    "identityToken": identity_token
}

mc_auth_response = requests.post(
    mc_auth_url,
    json=mc_auth_payload
)

mc_auth_data = mc_auth_response.json()

if mc_auth_response.status_code == 200:
    mc_access_token = mc_auth_data["access_token"]

    print("[🎉] Minecraft authentication successful!")
    print("Minecraft Access Token:", mc_access_token)
    print(mc_auth_data)

else:
    print("[❌] Minecraft authentication failed")
    print(mc_auth_response.status_code)
    print(mc_auth_data)

In [ ]:
# STEP 7: Check if the account owns Minecraft Java Edition and fetch the profile
profile_url = "https://api.minecraftservices.com/minecraft/profile"

profile_headers = {
    "Authorization": f"Bearer {mc_access_token}",
    "Content-Type": "application/json"
}

profile_response = requests.get(profile_url, headers=profile_headers)

if profile_response.status_code == 200:
    profile_data = profile_response.json()

    username = profile_data["name"]
    uuid = profile_data["id"]

    print("[🎉] Minecraft Java Edition ownership confirmed.")
    print("Minecraft Username:", username)
    print("Minecraft UUID:", uuid)

elif profile_response.status_code == 404:
    print("[❌] This account does not own Minecraft Java Edition or does not have a Java profile.")

else:
    print("[❌] Failed to fetch Minecraft profile.")
    print(profile_response.status_code)
    print(profile_response.text)

In [ ]:
# Dump into json
import json

data = {
    "username": username,
    "uuid": uuid,
    "token": mc_access_token,
    "clientid": CLIENT_ID,
    "auth_xuid": 0
}

with open("data.json", "w") as f:
    json.dump(data, f, indent=4)